# Sample Repositories Testing

This notebook tests the repositories related to the `Sample` lifecycle:
- `SampleRepository`
- `QualityControlRepository`
- `LogTemperatureRepository`

Since these entities have foreign key dependencies, we will first create the required core entities (`Patient`, `SampleType`, `Container`).

In [1]:
# Setup database and import required modules
from domain.database import Base, engine
from domain.config import DB_URL
from domain.repositories.unit_of_work import UnitOfWorkFactory

from domain.models import Patient, SampleType, Container, Sample, QualityControl, LogTemperature
from domain.repositories.patient_repository import PatientRepository
from domain.repositories.sample_type_repository import SampleTypeRepository
from domain.repositories.base_repository import BaseRepository
from domain.repositories.sample_repository import SampleRepository
from domain.repositories.quality_control_repository import QualityControlRepository
from domain.repositories.log_temperature_repository import LogTemperatureRepository

from datetime import date, datetime
import warnings
warnings.filterwarnings('ignore')


# Initialize Unit of Work Factory
uow_factory = UnitOfWorkFactory(DB_URL)

# Pre-requisite data setup
with uow_factory.create() as uow:
    # Creating prerequisites for Sample
    p = Patient(code="P-100", name="Test", lastname="Patient", birth_date=date(1990, 1, 1), active=True, test="Blood Test")
    st = SampleType(type_name="Plasma")
    c = Container(code="C-100", type_name="Vial")
    
    PatientRepository(uow.session).save(p)
    SampleTypeRepository(uow.session).save(st)
    BaseRepository[Container, int](uow.session, Container).save(c)
    uow.commit()
    
    # Save IDs to use later
    patient_id = p.id
    sample_type_id = st.id
    container_id = c.id
    print("Pre-requisite data (Patient, SampleType, Container) created successfully.")

✅ Configuration loaded: development -> sqlite:///C:\Users\Usuario\Desktop\biotrack\data\biotrack.db
Pre-requisite data (Patient, SampleType, Container) created successfully.


## 1. Sample Repository Tests

In [2]:
# Test CRUD and specific queries for SampleRepository
with uow_factory.create() as uow:
    sample_repo = SampleRepository(uow.session)

    # --- CREATE ---
    print("--- CREATE SAMPLES ---")
    s1 = Sample(
        code="SMP-001", 
        volume=10.5, 
        extraction_date=date(2024, 1, 10), 
        status="pending", 
        id_patient=patient_id, 
        id_sample_type=sample_type_id, 
        id_container=container_id
    )
    s2 = Sample(
        code="SMP-002", 
        volume=5.0, 
        extraction_date=date(2024, 2, 15), 
        status="analyzed", 
        id_patient=patient_id, 
        id_sample_type=sample_type_id, 
        id_container=container_id
    )
    
    sample_repo.save(s1)
    sample_repo.save(s2)
    # Assign protocol to sample
    from domain.repositories.protocol_repository import ProtocolRepository
    protocol_repo = ProtocolRepository(uow.session)
    prot = protocol_repo.get_by_code("PR-001")
    if prot:
        s1.protocols.append(prot)
    uow.commit()
    print(f"Created: {s1.code} (Status: {s1.status}, Volume: {s1.volume})")
    print(f"Created: {s2.code} (Status: {s2.status}, Volume: {s2.volume})\n")

    # --- READ BY ID & GET ALL ---
    print("--- READ OPERATIONS ---")
    s_read = sample_repo.get_by_id(s1.id)
    print(f"get_by_id({s1.id}): {s_read.code}")
    print(f"get_all(): Total samples = {len(sample_repo.get_all())}\n")

    # --- SPECIFIC QUERIES ---
    print("--- SPECIFIC QUERIES ---")
    print(f"get_by_code('SMP-001'): {sample_repo.get_by_code('SMP-001').volume} ml")
    
    status_pending = sample_repo.get_by_status("pending")
    print(f"get_by_status('pending'): {len(status_pending)} sample(s) found.")
    
    date_range = sample_repo.get_by_extraction_date_range(date(2024, 1, 1), date(2024, 1, 31))
    print(f"get_by_extraction_date_range(Jan 2024): {len(date_range)} sample(s) found.")
    
    patient_samples = sample_repo.get_by_patient_code("P-100")
    print(f"get_by_patient_code('P-100'): {len(patient_samples)} sample(s) found.\n")

    # --- UPDATE ---
    print("--- UPDATE ---")
    s1.status = "in_process"
    uow.commit()
    print(f"Updated SMP-001 status to: {sample_repo.get_by_id(s1.id).status}\n")
    
    # Note: We will NOT delete here because QualityControl and LogTemperature will need these samples.

--- CREATE SAMPLES ---
Created: SMP-001 (Status: pending, Volume: 10.5)
Created: SMP-002 (Status: analyzed, Volume: 5.0)

--- READ OPERATIONS ---
get_by_id(1): SMP-001
get_all(): Total samples = 2

--- SPECIFIC QUERIES ---
get_by_code('SMP-001'): 10.5 ml
get_by_status('pending'): 1 sample(s) found.
get_by_extraction_date_range(Jan 2024): 1 sample(s) found.
get_by_patient_code('P-100'): 2 sample(s) found.

--- UPDATE ---
Updated SMP-001 status to: in_process



## 2. Quality Control Repository Tests

In [3]:
# Test QualityControlRepository
with uow_factory.create() as uow:
    qc_repo = QualityControlRepository(uow.session)
    sample_repo = SampleRepository(uow.session)
    
    # Fetch sample for FK relation
    s1 = sample_repo.get_by_code("SMP-001")

    print("--- CREATE QUALITY CONTROL ---")
    qc1 = QualityControl(purity=95.5, concentration=12.4, result="approved", id_sample=s1.id)
    qc_repo.save(qc1)
    uow.commit()
    print(f"Created QC for {s1.code}: Purity {qc1.purity}%, Result {qc1.result}\n")

    print("--- SPECIFIC QUERIES ---")
    qc_by_sample = qc_repo.get_by_sample_code("SMP-001")
    print(f"get_by_sample_code('SMP-001'): Result = {qc_by_sample.result}")
    
    approved_qcs = qc_repo.get_by_result("approved")
    print(f"get_by_result('approved'): {len(approved_qcs)} record(s)")
    
    low_purity = qc_repo.get_below_purity(90.0)
    print(f"get_below_purity(90.0): {len(low_purity)} record(s)\n")
    
    # Eager Load Test
    qc_eager = qc_repo.get_with_sample(qc1.id)
    print(f"get_with_sample({qc1.id}): Sample Code eagerly loaded -> {qc_eager.sample.code}\n")

    print("--- DELETE ---")
    qc_repo.delete(qc1)
    uow.commit()
    print(f"Deleted QC record. Total QC records: {qc_repo.count()}")

--- CREATE QUALITY CONTROL ---
Created QC for SMP-001: Purity 95.5%, Result approved

--- SPECIFIC QUERIES ---
get_by_sample_code('SMP-001'): Result = approved
get_by_result('approved'): 1 record(s)
get_below_purity(90.0): 0 record(s)

get_with_sample(1): Sample Code eagerly loaded -> SMP-001

--- DELETE ---
Deleted QC record. Total QC records: 0


## 3. Log Temperature Repository Tests

In [4]:
# Test LogTemperatureRepository
with uow_factory.create() as uow:
    log_repo = LogTemperatureRepository(uow.session)
    sample_repo = SampleRepository(uow.session)
    
    s1 = sample_repo.get_by_code("SMP-001")

    print("--- CREATE LOG TEMPERATURES ---")
    log1 = LogTemperature(temperature=-80.5, id_sample=s1.id)
    log2 = LogTemperature(temperature=-79.0, id_sample=s1.id)
    log_repo.save(log1)
    log_repo.save(log2)
    uow.commit()
    print(f"Created Logs for {s1.code}: {log1.temperature}°C and {log2.temperature}°C\n")

    print("--- SPECIFIC QUERIES ---")
    logs = log_repo.get_by_sample_code("SMP-001")
    print(f"get_by_sample_code('SMP-001'): {len(logs)} log(s) found")
    
    latest_log = log_repo.get_latest_by_sample_code("SMP-001")
    print(f"get_latest_by_sample_code('SMP-001'): {latest_log.temperature}°C")
    
    out_of_range = log_repo.get_out_of_range(min_temp=-85.0, max_temp=-70.0)
    print(f"get_out_of_range(-85.0, -70.0): {len(out_of_range)} anomalies found (0 is expected here)")

    print("\n--- DELETE ---")
    log_repo.delete(log1)
    log_repo.delete(log2)
    uow.commit()
    print(f"Deleted Log records. Total Log records: {log_repo.count()}")

--- CREATE LOG TEMPERATURES ---
Created Logs for SMP-001: -80.5°C and -79.0°C

--- SPECIFIC QUERIES ---
get_by_sample_code('SMP-001'): 2 log(s) found
get_latest_by_sample_code('SMP-001'): -80.5°C
get_out_of_range(-85.0, -70.0): 0 anomalies found (0 is expected here)

--- DELETE ---
Deleted Log records. Total Log records: 0
